<a href="https://colab.research.google.com/github/SharanSapkota/Backend-form-validation-API-node-/blob/master/Copy_of_exercise_3_Ram_sapkota_2509739.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Affective Computing - Programming Assignment 3

### Objective

Your task is to use the feature-level method to combine facial expression features and audio features. A multi-modal emotion recognition system is constructed to recognize happy versus sadness facial expressions (binary-class problem) by using a classifier training and testing structure.

The original data is collected from ten actors acting happy and sadness behaviors.
* Task 1: Subspace-based feature fusion method: In this case, z-score normalization is utilized. Please read “Fusing Gabor and LBP feature sets for kernel-based face recognition” and learn how to use subspace-based feature fusion method for multi-modal system.

* Task 2: Based on Task 1, create a Leave-One-Subject-Out (LOSO) cross-validation to estimate the performance more reliably.

To produce emotion recognition case, Support Vector Machine (SVM) classifiers are trained.  50 videos from 5 participants are used to train the emotion recognition systems by using spatiotemporal features. The rest of the data (50 videos) are used to evaluate the performances of the trained recognition systems.

## Task 1. Subspace-based method  
Please read “Fusing Gabor and LBP feature sets for kernel-based face recognition” and apply their framework for the exercise. We use Support Vector Machine (SVM) with linear kernel for classification. As opposed to using Gabor features we are using the prosodic features from the last exercise.


### Setting up the environment

First, we need to import the basic modules for loading the data and data processing

In [ ]:
import sys
sys.path.append('../')
from skimage import io
from skimage import transform
from skimage import color
from skimage import img_as_ubyte
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import sklearn
import scipy.io as sio

### Loading data  <font color='red'>(0.5 point)</font>

We load the facial expression data (training data, training class, testing data, testing class) and audio data (training data, testing data)

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving lab3_data.mat to lab3_data.mat


In [ ]:
mdata = sio.loadmat('lab3_data.mat')

#Facial expression training and testing data, training and testing class
training_data = mdata['training_data']
testing_data = mdata['testing_data']
training_class = mdata['training_class'].ravel()
testing_class = mdata['testing_class'].ravel()

#Audio training and testing data
training_data_proso = mdata['training_data_proso']
testing_data_proso = mdata['testing_data_proso']

### Extract the subspace for facial expression features and audio features <font color='red'>(3 points)</font>
Extract the subspace for facial expression features and audio features using principal component analysis through using **PCA class**.
The `reduced_dim` is the dimensionality of the reduced subspace.
Set `reduced_dim` to 20 and 15 for facial expression features and audio features, respectively. Normalization should be done subject wise. The test data should be normalized with the values from the training data.
For concatenating the features use the __[`np.concatenate()`](https://docs.scipy.org/doc/numpy/reference/generated/numpy.concatenate.html)__ function.

You will implement the PCA class with two methods, **fit** and **transform**. The **fit** method takes one input array with no return values and the **transform** method takes one input array and returns a transformed array with dimensions. Use (__[`numpy.linalg.svd`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html)__) for singular values extraction.

In [ ]:
class PCA:
    """Principal component analysis (PCA).
    (docstring unchanged)
    """
    def __init__(self, n_components: int, whiten: bool = False) -> None:
        self.n_components = n_components
        self.whiten = whiten
        self.selected_components = None
        self.mean = None

    def fit(self, X: np.ndarray) -> None:
        #Step 1: Find the mean, and center the data
        self.mean = np.mean(X, axis=0)
        X = X - self.mean

        #Step2:  Find the Covariance
        cov = np.cov(X, rowvar=False)

        #Step 3: Apply SVD and choose the components, make the hermitian argument True.
        U, S, Vh = np.linalg.svd(cov, hermitian=True)
        self.selected_components = U[:, :self.n_components]
        # choose the singular values of diagnal matrix
        self.explained_variance = S[:self.n_components]

    def transform(self, X: np.ndarray) -> np.ndarray:
        # Center the data
        X = X - self.mean

        # Step 4: Choose and transform the features
        X_transformed = X @ self.selected_components
        if self.whiten:
            # Normalize the transform features
            X_transformed /= np.sqrt(self.explained_variance)
        return X_transformed

In [ ]:
#from sklearn.decomposition import PCA
from scipy import stats

#Set Reduced_dim for facial expression features and audio features, respectively.
reduced_dim_v = 20
reduced_dim_a = 15

#Extract the subspace for facial expression features though PCA.
pca_v = PCA(reduced_dim_v)
pca_v.fit(training_data)

#Transform training_data and testing data respectively
train_v = pca_v.transform(training_data)
test_v = pca_v.transform(testing_data)

#Extract the subspace for audio features though PCA
pca_a = PCA(reduced_dim_a)
pca_a.fit(training_data_proso)

#Transform the training_data and testing_data respectively
train_a = pca_a.transform(training_data_proso)
test_a = pca_a.transform(testing_data_proso)

#Normalize the features
# z-score using the training statistics only (test uses the training mean/std)
mean_v, std_v = train_v.mean(axis=0), train_v.std(axis=0)
mean_a, std_a = train_a.mean(axis=0), train_a.std(axis=0)

train_v = stats.zscore(train_v, axis=0)
test_v = (test_v - mean_v) / std_v
train_a = stats.zscore(train_a, axis=0)
test_a = (test_a - mean_a) / std_a

#Concatenate the transformed training data of facial expression features and audio features together
combined_train = np.concatenate((train_v, train_a), axis=1)

#Concatenate the transformed testing data of facial expression features and audio features together
combined_test = np.concatenate((test_v, test_a), axis=1)

### Question 1. Why is PCA used? Why not just concatenate the extracted features without PCA? <font color='red'>(0.5 point)</font>

### Your answer:



1.  Too many features, too few samples. The face data has 708 features but only 50 training samples, so the model can overfit. PCA keeps only the most useful features (20 for face, 15 for audio).
2.   Without PCA, we would join 708 face features with 15 audio features. The face part would be so big that audio would barely matter. PCA makes both parts similar in size, so both help the decision.
3. PCA keeps the important parts of the data, which makes the model faster and more accurate.



### Feature classification <font color='red'>(0.5 point)</font>
Use the __[`SVM`](http://scikit-learn.org/stable/modules/svm.html)__ function to train Support Vector Machine (SVM) classifiers.
Construct a SVM using the combined training data and linear kernel. The `training_class` group vector contains the class of samples: 1 = happy, 2 = sadness, corresponding to the rows of the training data matrices.

Then, calculate average classification performances for both training and testing data. The correct class labels corresponding with the rows of the training and testing data matrices are in the variables ‘training_class’ and ‘testing_class’, respectively.

In [ ]:
from sklearn import svm

# Train SVM classifier
clf = svm.SVC(kernel='linear')
clf.fit(combined_train, training_class)

#The prediction results
train_pred = clf.predict(combined_train)
test_pred = clf.predict(combined_test)

#Calculate and print the training accuracy and testing accuracy.
train_acc = np.mean(train_pred == training_class) * 100
test_acc = np.mean(test_pred == testing_class) * 100
print("Training accuracy: %.2f%%" % train_acc)
print("Testing accuracy: %.2f%%" % test_acc)

Training accuracy: 100.00%
Testing accuracy: 98.00%


### <font color='red'>(0.5 point)</font>
Compute the confusion matrices using __[`sklearn.metrics.confusion_matrix()`](http://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)__function for both the training data and testing data.


In [ ]:
from sklearn.metrics import confusion_matrix

cm_train = confusion_matrix(training_class, train_pred)
cm_test = confusion_matrix(testing_class, test_pred)

print("Training confusion matrix:\n", cm_train)
print("Testing confusion matrix:\n", cm_test)

Training confusion matrix:
 [[25  0]
 [ 0 25]]
Testing confusion matrix:
 [[25  0]
 [ 1 24]]


### Question 2. Compare the results from all the the different methods from assignments 1, 2 and 3. What method performed the best? What was the worst? Hypothesize as to why certain methods performed better than others. <font color='red'>(0.5 point)</font>

### Your answer:

Assignment 3 (fusion of face and prosody) performed best, with 98% testing accuracy (100% training), followed by MFCC SVM from Assignment 2 with 84%, then face-only from Assignment 1 with 72%. The worst was the prosody SVM from Assignment 2, with only 62% testing accuracy.

Fusion likely did best because face and voice carry different information about emotion, so when one is unclear the other helps. PCA and z-score normalization also kept the 708 face features from overwhelming the 15 prosody features and reduced overfitting on the small dataset. Prosody alone did worst because its few features (pitch, energy and similar) vary a lot between actors and don't separate happy from sad well on new people. MFCC did better than prosody because it describes the sound of the voice in more detail. Face-only did worse than fusion probably because it overfitted (88% training vs. 72% testing) and wrongly labeled many sad videos as happy.

## Task 2:
For a more reliable evaluation, often the Leave-One-Subject-Out (LOSO) cross-validation is used instead of the common train-test split. Cross-validation gives us a more reliable measure of the performance as all of the data is used for both training and testing. LOSO is used as emotions are highly dependent on the subject. By using LOSO, we guarantee that a subject is always in either the training or testing data and not in both.

* Join the training/testing data matrices and the class vectors. Combine also the ‘training_data_personID’ and ‘testing_data_personID’ vectors.

* Assume we have a total of $n$ subjects. Now, we will create a total of $n$ folds (loops), where each folds' training set contains the data from $n-1$ subjects and the testing set consists of only $1$ subject.

* Follow the steps taken in the first task: project the data to a subspace using PCA, conatenate the audio and video features together, train an SVM and finally evaluate the performance.

* The solution should be able to generalize over different numbers of subjects and samples, *e.g.*, a dataset may have 24 subjects, where subject1 has 4 samples and subject2 has 32 samples.

### <font color='red'>(0.5 point)</font>

In [ ]:
mdata = sio.loadmat('lab3_data.mat')

#Combine the training data, testing data,label and persion ID for video and audio respectively, in order to get the whole dataset.
lbp_data = np.concatenate((mdata['training_data'], mdata['testing_data']), axis=0)
proso_data = np.concatenate((mdata['training_data_proso'], mdata['testing_data_proso']), axis=0)

labels = np.concatenate((mdata['training_class'].ravel(), mdata['testing_class'].ravel()))
subjects = np.concatenate((mdata['training_personID'].ravel(), mdata['testing_personID'].ravel()))

#Get the number of the subject
subject_ids = np.unique(subjects)

#Print the shapes and the list of subject_ids for a sanity check
print(lbp_data.shape, proso_data.shape, labels.shape, subjects.shape)
print(subject_ids, len(subject_ids))

(100, 708) (100, 15) (100,) (100,)
[ 1  2  3  4  5  7  8  9 10 12] 10


### <font color='red'>(3 points)</font>

In [ ]:
accuracies = []
#Loop over each subject
for subject_id in subject_ids:
    #Create a boolean array for the training and testing set indices
    train_idx = subjects != subject_id
    #Similar for the test_idx, True indicates the position of the current subject_id
    test_idx = subjects == subject_id

    #Create the training and testing sets for lbp, proso and labels
    lbp_train, lbp_test = lbp_data[train_idx], lbp_data[test_idx]
    proso_train, proso_test = proso_data[train_idx], proso_data[test_idx]
    y_train, y_test = labels[train_idx], labels[test_idx]

    pca_v = PCA(n_components=20, whiten=True)
    pca_a = PCA(n_components=15, whiten=True)

    #Fit the PCAs with the training data
    pca_v.fit(lbp_train)
    pca_a.fit(proso_train)

    #Transform both the training and testing data with the PCA
    train_v = pca_v.transform(lbp_train)
    test_v = pca_v.transform(lbp_test)
    train_a = pca_a.transform(proso_train)
    test_a = pca_a.transform(proso_test)

    #Concatenate the features together
    X_train = np.concatenate((train_v, train_a), axis=1)
    X_test = np.concatenate((test_v, test_a), axis=1)

    #Create a linear SVM and train it
    clf = svm.SVC(kernel='linear')
    clf.fit(X_train, y_train)

    #Calculate the accuracy for the testing data and add it to the list of accuracies
    acc = np.mean(clf.predict(X_test) == y_test)
    accuracies.append(acc)

#Calculate the average of the accuracies. Print both the list of accuracies and the average
print("Accuracies per subject:", accuracies)
print("Average accuracy: %.2f%%" % (np.mean(accuracies) * 100))

Accuracies per subject: [np.float64(0.9), np.float64(0.8), np.float64(1.0), np.float64(0.9), np.float64(0.9), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(0.8), np.float64(1.0)]
Average accuracy: 93.00%


### Question 3. The accuracy of LOSO (0.93) is lower than the accuracy achieved by the train-test split (0.98) in task 1. Hypothesize as to why the two are different. Which one is better for evaluation?  <font color='red'>(0.5 point)</font>

### Your answer:



1.   It had only 50 samples from 5 people, so one lucky or unlucky split changes the result a lot. Those 5 test people may simply be easier to classify than the average person, which would make 98% too optimistic.
2.   All 10 subjects get a turn as the unseen test person, including harder ones (the 80% subjects). The 93% average therefore reflects the whole group instead of one favorable split.


1.   Each fold trains on 9 subjects instead of 5, so you might expect it to do better. It is still lower because it covers all the harder subjects.
2.   Each actor expresses happy and sad differently, so a model can struggle on a new person. Accuracy varies from 80% to 100% across subjects, which shows this.





### Question 4. In PCA why `whiten` parametere is better and why it replaces the normalization?  <font color='red'>(0.5 point)</font>

### Your answer:

Whitening is better because it divides each PCA component by the square root of its variance, which gives every component the same scale (variance 1) and keeps the components uncorrelated, while z-score normalization only fixes the scale. It replaces the separate normalization step because it does the same job, making all features equal in scale so none dominates the SVM, but inside the PCA itself and using only the training data's variances, so the test data is scaled with training values and nothing leaks from the test set.